# Analysing the MoMA Collection Using Data Science and Art-History

I've recently found myself drawn to machine learning and AI, and for a reason I ignore, these tools applied to art historical studies fascinates me. A part of me wonders that art, mouvements and timelines are so in-tangible and arbirtry that a machine surely would not be able to be able to classify conbtemporary art correctly. Well, it may be able to, can it do so precisely and accurately while taking into account a wast number of non-numerical data: the artist biography, cultural influences, blur frontiers between genres. 

The object of the analysis is not to answer that question. This report is an exploration of the structured data that is the MoMA's collection. 

Through conductiung this analysis I gained data science skills, programming knowlegde and general insight on how to conduct a data science project. 

This project will focus on being exhaustive and didactic making it accessible to art historian and new developers alike. I apologize in advance to any developer who might find this to verbose.

## A timeline of mistakes

I initialy started this project thinking that I could create a RNN(Recurrent Neural Network) by simply parsing the MoMA's CSV's, telling the model to learn from: mediums, date created and labelseg impressionism. 

That would have been all good and well, albeit the first initial mistake I commited: not know how my data was structured. As a matter of fact, I was quickly confronted to a simple reality: data science (basic knowledge) is almost necessary to be able to conduct a larger machine learning project. 

Indeed, without a proper undersanding of the data I was looking at, I could not ask proper questions. As a matter of fact, I had and idea of a question, and a rather silly one at that: "If this data set was given to AI to understand what Contemporary Art *is* would would be it's output?". Think like our civilisation has long disapeared and as part of a last effort to save humanities legacy, this dataset was selected (amongst others) to be preserved for millions of years. ((how ethnocentric of me)).

Despite the silliness of this question, I still think it is one worth asking. Even though I will not be answering it right now, I believe it is essential for the reader to understand my motivations behind this data analysis project. 

Going back to my mistake and how I corrected it, its seemed clear to me that even if I wasn't going to be able to answer this complex question, I could make sense of one idea: maybe I could understand the evolution of contemporary art through this dataset and inch towards an answer "what is contemporary art?" 

## A Surface exploration of the MoMA dataset

The [MoMA dataset or collection](https://github.com/MuseumofModernArt/collection) I will be refering to is an open source repository of CSV's created by the MoMA. It is publicly accessible to anyone who knows how to use github.

The github repository provides us a little description of the collection: 
*"The Museum of Modern Art (MoMA) acquired its first artworks in 1929, the year it was established. Today, the Museum’s evolving collection contains almost 200,000 works from around the world spanning the last 150 years. The collection includes an ever-expanding range of visual expression, including painting, sculpture, printmaking, drawing, photography, architecture, design, film, and media and performance art."* 

The dataset is composed of two main CSV's: Artists and Artworks which respectively represent All Artosts in collection and all artworks in the collection. 

The collection it's self is visible [here](https://www.moma.org/collection/)

And the uses of this dataset have been recorded in a [medium entry](https://medium.com/@foe/here-s-a-roundup-of-how-people-have-used-our-data-so-far-80862e4ce220#.f6272outn). Unfortunatley some of the links contained in this post have been affected by bit-rot. It is also worth metionning that this was originally posted in 2015 and seems to have received few updates since (as of Wed Aug 6th). 

The online presence of the use of this dataset also extends to a [Kaggle](https://www.kaggle.com/datasets/momanyc/museum-collection) entry which contains some insightduk data analysis albeit lacks of a art-historical analysis. I would specifically artact eyes to [this entry](https://www.kaggle.com/code/fangya/moma-eda-cloud-tour-gans) which explores various information about teh dataset and provided a good entry point for me to understand what was possible. 

Finally, i will also highkist a very insightful [article](https://towardsdatascience.com/using-momas-collection-dataset-to-visualize-whose-stories-are-missing-76a8960a33c2/) on the representation - or lack of - marginalzied communities through the dataset which was materialized into a [data science art installation](https://cdn.embedly.com/widgets/media.html?src=https%3A%2F%2Fplayer.vimeo.com%2Fvideo%2F524606006%3Fapp_id%3D122963&dntp=1&display_name=Vimeo&url=https%3A%2F%2Fvimeo.com%2F524606006&image=https%3A%2F%2Fi.vimeocdn.com%2Fvideo%2F1086436662_1280.jpg&key=a19fcc184b9711e1b4764040d3dc5c07&type=text%2Fhtml&schema=vimeo). The conclusions are clear: the MoMA's collectio is composed of a majority of white male artists. 

This conclusion raises the question, how can we make sense of this conclusion from a curatorial and museo grahipcal perspective? Have things changed? What dictates the MoMA's acquisition policy? Has there been undelying trends and anomalies in the recent years ignifiying a change in philosphy? and finally, is data science an effective tool to help us answer these questions. 


In [6]:
# import dataframes from CSV's 
from data.scripts.utils.data_frames import artists, artworks

print(f"Artworks has {len(artworks.columns.values)}{artworks.columns.values}"), print(f"Artists has {len(artists.columns.values)} {artists.columns.values}")

Artworks has 30['Title' 'Artist' 'ConstituentID' 'ArtistBio' 'Nationality' 'BeginDate'
 'EndDate' 'Gender' 'Date' 'Medium' 'Dimensions' 'CreditLine'
 'AccessionNumber' 'Classification' 'Department' 'DateAcquired'
 'Cataloged' 'ObjectID' 'URL' 'ImageURL' 'OnView' 'Circumference (cm)'
 'Depth (cm)' 'Diameter (cm)' 'Height (cm)' 'Length (cm)' 'Weight (kg)'
 'Width (cm)' 'Seat Height (cm)' 'Duration (sec.)']
Artists has 9 ['ConstituentID' 'DisplayName' 'ArtistBio' 'Nationality' 'Gender'
 'BeginDate' 'EndDate' 'Wiki QID' 'ULAN']


(None, None)

As we can see, the artworks csv is composed of 30 columns ranging from identifiers to material characteristics about the artwork. The artist csv is composed of 9 columns that are quite straght forward. 

Since my analysis will be focusing on artworks, I was immediately drawn to this dataset over the artists data. 

Before diving into teh artworks, let's get a better understanding of the whole collection. 

In [8]:
from data.scripts.artists import ArtistCols
from data.scripts.artworks import ArtworkCols
from data.scripts.utils.utils import clean_years

artist_nationalities = artists[ArtistCols.Nationality.value].nunique()
departments = artworks[ArtworkCols.Department.value].nunique()
classifications = artworks[ArtworkCols.Classification.value].nunique()
mediums = artworks[ArtworkCols.Medium.value].nunique()

entries_created_at_year = (
    clean_years(artworks[ArtworkCols.Date.value]).dropna().sort_values()
)

entries_acquired_at_year = (
    artworks[ArtworkCols.DateAcquired.value].dropna().sort_values()
)

print("Information about the current MOMA collection:\n")
print(
    f"The MOMA counts {len(artworks)} artworks by {len(artists)} artists from {artist_nationalities} nationalities\n"
)
print(
    f"The MOMA currently counts {departments} departments with {classifications} different classifications and {mediums} mediums\n"
)

print(
    f"The earliest artwork dates back to: {entries_created_at_year.iloc[0]} and latest: {entries_created_at_year.iloc[len(entries_created_at_year) - 1]}\n"
)

print(
    f"The earliest acquired entry dates back to: {entries_acquired_at_year.iloc[0]} and latest: {entries_acquired_at_year.iloc[len(entries_acquired_at_year) - 1]}\n"
)


Information about the current MOMA collection:

The MOMA counts 159716 artworks by 15720 artists from 139 nationalities

The MOMA currently counts 8 departments with 40 different classifications and 22944 mediums

The earliest artwork dates back to: 1768 and latest: 2025

The earliest acquired entry dates back to: 1929-11-19 and latest: 2025-06-10



As we can see, the MoMA's collection is quite large, which by the way, makes it a greate dataset to train neural networks. 

Moving forward, these numbers help me identify which fields which were the most relevent for the analysis of the artworks dataset. 

As we can see, there are three main categories that are relevant for an artwork analysis: departments, classifications and mediums. We can see here two striking information:
- there is a vast quantity (22944) of uniaue mediums, which means that no harmonisation has been made on the artwork mediums which remain precise and spceific to each artwork. 
- a "mouvement" category is absent. I was indeed expecting a classification by art mouvements such as futurism, impressionism etc 

In addition to these first observations, it's worth mentionning the year range (1768 - 2025) of artworks in the MoMA's collection.

Finally, we'll note that the museum started acquiring artworks in November 1929, the same month and year the museum officially opened. 


